(dkist:examples:visp-extents)=

# Showing the Field of View of VISP on AIA


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.time import Time

import sunpy.map
from sunpy.net import Fido, attrs as a
from sunpy.visualization import drawing

import dkist
import dkist.net
from dkist.data.sample import VISP_L1_KMUPT

```{note}
This example requires `sunpy>=6.1` for the {obj}`sunpy.visualization.extent` function.
Also if you installed dkist with pip you may need to run ``pip install sunpy[all]``, to install the dependancies for sunpy's map.
```


## Obtaining some data

In this example we will use the VISP sample dataset [L1_KMUPT](https://dkist.data.nso.edu/datasetview/L1_KMUPT).
If you want to replace this dataset with your own dataset, see <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/howto_guides/sample_data.html#dkist-howto-guide-sample-data" target="_blank" style="text-decoration: underline">Downloading the Sample Data with Globus</a>.

Let's load the data with <a href="https://docs.dkist.nso.edu/projects/python-tools/en/stable/api/dkist.load_dataset.html#dkist.load_dataset" target="_blank" style="text-decoration: underline">`dkist.load_dataset`</a>:


In [ ]:
ds = dkist.load_dataset(VISP_L1_KMUPT)
ds

Extract the two pixel axes which are correlated to the spatial dimensions, (raster scan step number and spatial along slit).

This VISP dataset only has one raster, if you are using a dataset with repeated rasters, you can either pick one or you can iterate over all the raster repeats to plot the field of view for each one.


In [ ]:
spatial_slice = ds[0, :, 0, :]
spatial_slice

Extract the times of all the raster positions


In [ ]:
times = spatial_slice.axis_world_coords("time")[0]

And then download the AIA image closest to the time of the first raster position.


In [ ]:
results = Fido.search(a.Instrument.aia, a.Wavelength(171*u.AA), a.Time(times[0], times[-1], times[0]))
results

In [ ]:
aia_files = Fido.fetch(results, site="NSO")

Now we load the downloaded AIA file into a {obj}`sunpy.map.AIAMap` object.


In [ ]:
aia = sunpy.map.Map(aia_files)

In [ ]:
aia

Finally we can use a combination of <a href="https://docs.sunpy.org/en/stable/generated/api/sunpy.map.GenericMap.html#sunpy.map.GenericMap.plot" target="_blank" style="text-decoration: underline">`sunpy.map.GenericMap.plot`</a> and <a href="https://docs.sunpy.org/en/stable/generated/api/sunpy.visualization.drawing.extent.html#sunpy.visualization.drawing.extent" target="_blank" style="text-decoration: underline">`sunpy.visualization.drawing.extent`</a> to plot the extent of the VISP raster.


In [ ]:
fig = plt.figure(figsize=(8,8))
ax = fig.add_subplot(projection=aia)

aia.plot(axes=ax)

drawing.extent(ax, spatial_slice.wcs)

# Zoom in on the VBI region using pixel coordinates
_ = ax.axis((1500, 2000, 1900, 2400))